In [24]:
import pandas as pd
import numpy as np

PRO01_CSV_PATH = '../pro01.csv'

df = pd.read_csv(PRO01_CSV_PATH, encoding='cp949') #utf-8 or cp949

df

,연도,월,분기,청코드,내외항구분,수출입구분명,시설코드,시설명,부두구분명,아외국구분,적공구분,컨테이너수(10피트),컨테이너수(20피트),컨테이너수(40피트),컨테이너수(기타),전체개수,전체물동량
0,2024,3,1,신항,외항,수입,6,신항 W 정박지,일반부두,외국선,적컨,0,3,8,0,11,19.00
1,2024,4,2,신항,외항,수입,6,신항 W 정박지,일반부두,아국선,적컨,0,9,3,0,12,15.00
2,2024,11,4,신항,외항,수입환적,7,신항 U 정박지,일반부두,아국선,공컨,0,1,0,0,1,1.00
3,2024,11,4,신항,외항,수입,7,신항 U 정박지,일반부두,아국선,공컨,0,57,2,0,59,61.00
4,2024,11,4,신항,외항,수입환적,7,신항 U 정박지,일반부두,아국선,적컨,0,31,100,0,131,231.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2404,2024,7,3,북항,외항,수입환적,8,자성대 부두,컨테이너부두,외국선,적컨,0,8769,6434,0,15203,21637.00
2405,2024,7,3,북항,외항,수출,8,자성대 부두,컨테이너부두,외국선,적컨,0,7719,6438,2,14159,20599.50
2406,2024,7,3,북항,외항,수입,8,자성대 부두,컨테이너부두,외국선,적컨,0,7818,7253,0,15071,22324.00
2407,2024,11,4,북항,외항,수출,8,자성대 부두,컨테이너부두,외국선,적컨,0,2519,2018,3,4540,6561.75


In [25]:
pd.DataFrame({'자료형': df.dtypes.astype('str'),
              '비결측수': df.notna().sum(),
              '결측수': df.isna().sum(),
              '결측률(%)': df.isna().mean()*100,
              '고유값 수': df.nunique(dropna=True)})

,자료형,비결측수,결측수,결측률(%),고유값 수
연도,int64,2409,0,0.0,1
월,int64,2409,0,0.0,12
분기,int64,2409,0,0.0,4
청코드,str,2409,0,0.0,3
내외항구분,str,2409,0,0.0,1
수출입구분명,str,2409,0,0.0,4
시설코드,int64,2409,0,0.0,19
시설명,str,2409,0,0.0,30
부두구분명,str,2409,0,0.0,2
아외국구분,str,2409,0,0.0,2


In [26]:
col = df.columns.to_list
col

<bound method IndexOpsMixin.tolist of Index(['연도', '월', '분기', '청코드', '내외항구분', '수출입구분명', '시설코드', '시설명', '부두구분명',
       '아외국구분', '적공구분', '컨테이너수(10피트)', '컨테이너수(20피트)', '컨테이너수(40피트)',
       '컨테이너수(기타)', '전체개수', '전체물동량'],
      dtype='str')>

In [27]:
df.describe().T


,count,mean,std,min,25%,50%,75%,max
연도,2409.0,2024.000000,0.000000,2024.0,2024.0,2024.0,2024.00,2024.0
월,2409.0,6.593192,3.462982,1.0,4.0,7.0,10.00,12.0
분기,2409.0,2.529680,1.124354,1.0,2.0,3.0,4.00,4.0
시설코드,2409.0,11.799087,7.905031,1.0,8.0,9.0,14.00,31.0
컨테이너수(10피트),2409.0,2.574097,13.428490,0.0,0.0,0.0,0.00,132.0
컨테이너수(20피트),2409.0,2357.080531,3874.324901,0.0,44.0,407.0,3190.00,26277.0
컨테이너수(40피트),2409.0,3865.119967,7842.387127,0.0,84.0,889.0,3932.00,60733.0
컨테이너수(기타),2409.0,18.185969,54.339276,0.0,0.0,0.0,6.00,541.0
전체개수,2409.0,6242.960565,11470.764156,1.0,154.0,1381.0,7027.00,82897.0
전체물동량,2409.0,10129.522623,19308.235586,1.0,240.0,2316.5,11131.25,144125.0


In [28]:
data_cols = ['월', '수출입구분명','적공구분','컨테이너수(20피트)', '컨테이너수(40피트)'] #원하는 수치만 선택

project = df[data_cols]

In [29]:
project["전체total"] = (
    project["컨테이너수(20피트)"]
    + project["컨테이너수(40피트)"] * 2
)

project    #특이값(10피트,기타)제외한 합계 

# 2. 월별 전체 물동량
monthly_total = project.groupby("월")["전체total"].sum()

# 3. 월별 적컨 물동량
monthly_loaded = (
    project.loc[project["적공구분"] == "적컨"]
    .groupby("월")["전체total"]
    .sum()
)

# 4. 데이터프레임 생성
result = pd.DataFrame({
    "전체total": monthly_total,
    "적컨total": monthly_loaded
})

# 5. 적컨 비중 계산
result["적컨비중(%)"] = (
    result["적컨total"]
    .div(result["전체total"])
    .mul(100)
    .round(2)
)

result   #항구 효율

,전체total,적컨total,적컨비중(%)
월,,,
1,1983959,1633090,82.31
2,1869732,1562805,83.58
3,2136079,1730695,81.02
4,2037556,1691076,83.00
5,2088685,1751417,83.85
6,2082112,1741390,83.64
7,2098444,1760457,83.89
8,2044365,1712398,83.76
9,1869464,1564361,83.68


In [30]:
project["total"] = 0

mask = project["적공구분"] == "적컨"

project.loc[mask, "total"] = (
    project.loc[mask, "컨테이너수(20피트)"]
    + project.loc[mask, "컨테이너수(40피트)"] * 2
)



project["수출입구분명"].unique().tolist()  #수출입 구분 고유값 추출



['수입', '수입환적', '수출환적', '수출']

In [31]:
project.loc[project['수출입구분명'] ==('수입환적' or '수출환적')]

,월,수출입구분명,적공구분,컨테이너수(20피트),컨테이너수(40피트),전체total,total
2,11,수입환적,공컨,1,0,1,0
4,11,수입환적,적컨,31,100,231,231
7,3,수입환적,공컨,228,6675,13578,0
11,4,수입환적,공컨,556,1366,3288,0
15,8,수입환적,공컨,663,2065,4793,0
...,...,...,...,...,...,...,...
2390,10,수입환적,적컨,2754,1864,6482,6482
2394,2,수입환적,적컨,7707,6240,20187,20187
2398,6,수입환적,적컨,9291,6775,22841,22841
2402,11,수입환적,적컨,3128,2159,7446,7446


In [37]:

result = []
for i in project["월"].unique().tolist():
    month_sum = project.loc[project['월'] == i ].sum()
    month_sum["월"] = i
    result.append(month_sum)
    
monthly_sum = pd.DataFrame(result)
monthly_sum = monthly_sum.sort_values("월")
result = monthly_sum[['월','total']]


In [33]:
result_month = result.sort_values("월").set_index("월")

result_month["증감률(%)"] = (
    result["total"]
    .pct_change()
    .mul(100)
    .round(2)
)

result_month   #월별데이터분석,추세분석

,total,증감률(%)
월,,
1,1633090,-2.29
2,1562805,-3.20
3,1730695,-2.73
4,1691076,9.89
5,1751417,-4.30
6,1741390,-0.57
7,1760457,1.09
8,1712398,3.57
9,1564361,4.85


In [34]:
season_map = {
    1: "겨울", 2: "겨울",
    3: "봄", 4: "봄", 5: "봄",
    6: "여름", 7: "여름", 8: "여름",
    9: "가을", 10: "가을", 11: "가을",
    12: "겨울"
}

result_month["계절"] = result_month.index.map(season_map)

season_result = (
    result_month
    .groupby("계절")["total"]
    .sum()
    .to_frame()
    .reindex(["봄", "여름", "가을", "겨울"])
)

season_result              #계절성 분석

,total
계절,
봄,5173188
여름,5214245
가을,4947465
겨울,4940679


In [35]:
trans_result = project.pivot_table(
    index="월",
    columns="수출입구분명",
    values="total",
    aggfunc="sum",
    fill_value=0
)

trans_result["환적"] = trans_result["수입환적"] + trans_result["수출환적"]

trans_result = trans_result[["환적", "수입", "수출"]]

trans_result["총합"] = trans_result[["환적", "수입", "수출"]].sum(axis=1)

trans_result["환적비중(%)"] = (
    trans_result["환적"]
    .div(trans_result["총합"])
    .mul(100)
    .round(2)
)
trans_result.columns.name = None
trans_result #환적 비중

,환적,수입,수출,총합,환적비중(%)
월,,,,,
1,1023286,252830,356974,1633090,62.66
2,996903,215444,350458,1562805,63.79
3,1045215,287264,398216,1730695,60.39
4,1030694,284270,376112,1691076,60.95
5,1107243,272374,371800,1751417,63.22
6,1079810,284153,377427,1741390,62.01
7,1101034,270195,389228,1760457,62.54
8,1077122,277436,357840,1712398,62.90
9,985125,248707,330529,1564361,62.97


In [36]:
trans_result.describe()

,환적,수입,수출,총합,환적비중(%)
count,1.200000e+01,12.000000,12.000000,1.200000e+01,12.000000
mean,1.056747e+06,265534.416667,367350.333333,1.689631e+06,62.555833
std,4.105628e+04,20488.135349,18401.231991,6.958054e+04,0.996690
min,9.851250e+05,215444.000000,330529.000000,1.562805e+06,60.390000
25%,1.028842e+06,253208.750000,356571.250000,1.656308e+06,62.407500
50%,1.066236e+06,271284.500000,369079.000000,1.715727e+06,62.795000
75%,1.085116e+06,279115.250000,377545.000000,1.742238e+06,63.152500
max,1.107243e+06,287264.000000,398216.000000,1.760457e+06,63.790000
